# Fashion Intelligence: Problem Definition

## Executive summary

This project builds an end-to-end machine-learning system for a fashion catalogue. Given a
product image, the system predicts catalogue tags and retrieves visually similar products. The
assignment requires three classification tasks and one visual-search task, final models trained
from scratch, an independent final evaluation, official predictions, and a usable application.

| Task | Question answered | Required output |
|---|---|---|
| Task 1 | What type of item is shown? | `articleType` |
| Task 2 | Which season is the item intended for? | `season` |
| Task 3 | Who and what occasion is the item intended for? | separate `gender` and `usage` outputs |
| Task 4 | Which catalogue products look most similar? | ranked Top-K product IDs |

**Project decision.** One product ID is one reporting unit. Original and high-resolution images
are two input views of that product, not two independent examples. All tasks use the same saved
split and the same protected evaluation boundary.

**Sources of truth:** [assignment specification](../docs/COSC2753_2026B_Assignment%202.pdf),
[marking rubric](../rubrics/RUBRIC.md), and accepted [project decisions](../docs/decisions/README.md).

## 1. Real-world context and users

### 1.1 User needs

Fashion catalogues are large and change often. Manual tagging is slow and inconsistent. Customers
also need a visual way to find alternatives when they do not know the correct catalogue term.

### 1.2 Decisions supported

| User | System result | Decision or action |
|---|---|---|
| Catalogue operator | four predicted tags | review or complete a product record |
| Customer | Top-K similar products | inspect visually related catalogue items |
| ML team | metrics, errors, latency, and model size | choose a model suitable for deployment |

The models support these decisions. They do not replace human catalogue governance. Low-confidence
and visibly inconsistent cases should remain reviewable instead of being silently corrected.

## 2. Problem statement and prediction unit

### 2.1 Prediction unit

The prediction unit is one fashion product identified by `id`. A product may have an original
teacher image and a matched high-resolution image. Both views inherit the same split membership.
Training may encode both views, but metrics and search results collapse them back to one product.

### 2.2 Inputs available at prediction time

The standard model input is image pixels only. Catalogue metadata may define targets, audit data,
or retrieval relevance, but it is not supplied as a shortcut feature to the standard image
classifiers. The application must apply the same RGB conversion, aspect-preserving resize, padding,
and train-fitted normalization used during model development.

### 2.3 Outputs

Classification inference returns one value for each required field: `articleType`, `season`,
`gender`, and `usage`. Visual search returns ordered product IDs with similarity scores. Official
classification predictions keep the fixed schema `id,gender,articleType,season,usage`.

## 3. Machine-learning task definitions

### 3.1 Task 1 - Fashion item type classification

Task 1 is multiclass image classification over the official `articleType` vocabulary. It has a
severe long tail, so overall accuracy alone is not enough. The task owner must select and justify a
primary metric plus supporting class-support, confusion, and rare-class evidence before experiments.

### 3.2 Task 2 - Fashion season classification

Task 2 is multiclass image classification over the available `season` values. Season may be weakly
visible and correlated with item type. The investigation must test whether the model learns useful
visual evidence or relies on an article-type shortcut, and it must report ambiguity and calibration.

### 3.3 Task 3 - Gender and usage classification

Task 3 has two separate prediction targets because the official output requires separate `gender`
and `usage` columns. Separate models and a shared-representation model may be compared. Each target
keeps its own label mask, loss, metric, confusion matrix, and failure analysis so a gain in one head
cannot hide harm to the other.

### 3.4 Task 4 - Fashion visual search

Task 4 is a retrieval problem, not classification. A validation product acts as a query and eligible
training products form the development gallery. Query IDs, exact hashes, and product families must
not enter the gallery. The final output is ranked at product level; the Task 4 owner must investigate
and justify how image-view evidence is combined. Catalogue-derived relevance is an evaluation proxy,
not human visual-similarity truth.

## 4. Data roles and evaluation boundary

### 4.1 Raw labelled corpus

The teacher training CSV and matching images are read-only sources. Objective file failures, exact
duplicates, accepted near-duplicate links, and repeated product families are audited before model
development.

### 4.2 Development partitions

`train` fits preprocessing and models. `val` supports model comparison and provisional decisions.
Every notebook, model, retrieval index, and application reads `data/processed/splits.csv`; no other
split may be created.

### 4.3 Protected partitions

`holdout` remains label-sealed until final choices are frozen. `quarantine` contains data that must
not influence training or evaluation, including unsafe cross-role relationships and conflicting
exact-image labels.

### 4.4 Official prediction set

Official test images are unlabelled. They may enter label-free file and duplicate audits. A detected
cross-role duplicate may therefore quarantine a labelled product to prevent contamination, but official
test content must not influence label statistics, normalization, tuning, or the development gallery.

## 5. Success criteria and evaluation framework

### 5.1 Classification metric-selection criteria

Each classification owner must choose and justify one primary metric and useful supporting metrics
after reading the class balance, missing-label, and support evidence in Notebook 01. The selection must
avoid hiding weak classes, state valid denominators, and be frozen before model comparison. Candidate
evidence may include class-aware scores, per-class results, confusion patterns, calibration, product-level
uncertainty, variant consistency, and representative errors.

### 5.2 Retrieval metric-selection criteria

The Task 4 owner must select and justify ranking-quality, recovery, coverage, and operational metrics
that match the query, gallery, Top-K, and relevance definition. The choice must state how queries with
no relevant gallery item are handled and must be frozen before retrieval approaches are compared.

### 5.3 Final judgement criteria

Exact task metrics are intentionally not frozen in this notebook; each task owner records that decision
with evidence before experiments. The assignment specifies no numeric accuracy threshold, so this
project does not invent one before
baseline evidence exists. A final recommendation must be justified against fair baselines and include
independent holdout results, robustness, failure costs, model size, memory, and latency. A model is not
selected from one aggregate score alone.

## 6. Constraints and non-goals

### 6.1 Assignment and project constraints

- Final submitted models are fully trained from scratch. Pretrained systems may appear only as clearly
  separated comparison benchmarks.
- At least one suitable algorithm is investigated for each task, with broader comparisons required
  for a strong investigation.
- Every training run is registered with its data, configuration, metrics, timing, and checkpoint.
- The official prediction CSV format and row IDs are unchanged.
- Raw supplied data is not modified and is used only for the educational assignment.

### 6.2 Non-goals

This project does not build personalised recommendations, price ranking, stock prediction, automatic
catalogue relabelling, or a production-scale distributed search service. It demonstrates a defensible
end-to-end ML system and a usable application within the assignment scope.

## 7. Assumptions, risks, and failure costs

| Risk | Why it matters | Required response |
|---|---|---|
| Missing or corrupt image | breaks loading and removes class support | exclude only objective failures and keep an audit trail |
| Exact or near duplicate | inflates validation and can expose official test content | group or quarantine before the sole split |
| Long-tail labels | head classes can hide zero tail recall | class-aware metrics, support tables, imbalance experiment, rare-class errors |
| Weak or subjective tags | season, gender, and usage may not be fully visible | visual sanity samples, calibration, uncertainty, honest limitations |
| Target association | one output may act as a shortcut for another | association analysis and task-specific error slices |
| Two image variants | double counting overstates sample size | pair weights and product-level aggregation |
| Retrieval relevance proxy | metadata is not human similarity truth | choose justified metrics, report coverage and proxy limits |
| Domain shift | catalogue photos differ from user photos | degraded-input and external-evidence checks before deployment claims |

Classification errors can create bad catalogue filters. Retrieval errors can hide useful products or
show irrelevant ones. The application should expose confidence or similarity evidence and support
human review rather than presenting every output as certain. This is a product-design goal, not a claim
that the current assignment prototype already provides a complete review system.

## 8. End-to-end system design

### 8.1 Lifecycle

```text
Problem definition
  -> raw audit, hashing, reconciliation, and duplicate families
  -> one protected split and train-only preprocessing
  -> fair task-specific experiments and registered runs
  -> frozen provisional winners
  -> one independent holdout evaluation
  -> official predictions, saved models, and application
```

### 8.2 Notebook and production-code boundary

Notebooks tell the investigation story and show evidence. Reusable data loading, transforms, training,
evaluation, inference, and retrieval logic lives in `src/fashion/`. The application and prediction
script load the same saved preprocessing contract and checkpoints; they never depend on notebook state.

### 8.3 Traceability

Each reported result must trace to a run ID, configuration, split digest, input policy, metric version,
checkpoint, and figure or table. This makes comparisons reproducible and keeps the final judgement
auditable.

## 9. Deliverables and notebook handoff

### 9.1 Required deliverables

- A concise report explaining the investigation and final judgement.
- One frozen final method per task, with Task 3 covering both required outputs, and scripts that run them.
- Official classification predictions produced by the frozen final methods.
- Reproducible notebooks, source code, saved models, environment instructions, and an application.

### 9.2 Notebook reading order

1. `00_problem_definition.ipynb` - purpose, boundaries, metrics, and risks.
2. `01_data_preparation.ipynb` - raw audit through model-ready shared data.
3. `02_task1_article_type.ipynb` - Task 1 comparisons and provisional judgement.
4. `03_task2_season.ipynb` - Task 2 comparisons and provisional judgement.
5. `04_task3_gender_usage.ipynb` - Task 3 comparisons and provisional judgement.
6. `05_task4_visual_search.ipynb` - retrieval comparisons and provisional judgement.
7. `06_final_evaluation.ipynb` - frozen choices, holdout evaluation, and final deployment judgement.

## 10. Problem-definition completion gate

- [x] Real-world users and supported decisions are named.
- [x] Input, output, prediction unit, and four task types are defined.
- [x] Development, holdout, quarantine, and official-prediction roles are separated.
- [x] Metric-selection criteria, decision ownership, and broader final-judgement evidence are defined.
- [x] Assignment constraints, project rules, non-goals, assumptions, and risks are explicit.
- [x] The notebook and reusable-code boundary is fixed.
- [x] The next notebook has a clear input and responsibility.

**Next step.** Continue with `01_data_preparation.ipynb`. It must establish the immutable raw-file
record, image/tag reconciliation, duplicate and family controls, sole split, train-only evidence, and
shared preprocessing contract before any task model is trained.